# LeaseGuard Phase 6 — Compare T4 baselines

This notebook does **not** need a GPU. It reads the three candidate run files from Drive and selects the strongest practical unmodified base model using a lexicographic order: schema validity, extraction F1, answer status accuracy, evidence recall, speed, then lower peak VRAM.

It will not invent a blended LeaseGuard score. Official professional-benchmark claims still require LegalBench, CUAD, ContractNLI, or LegalBench-RAG with pinned evaluators.


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USER/LeaseGuard.git"
REPO_DIR = Path("/content/LeaseGuard")
DRIVE_ROOT = Path("/content/drive/MyDrive/leaseguard")
REPORT_DIR = DRIVE_ROOT / "reports" / "baselines"
OUTPUT = REPORT_DIR / "phase6_comparison.json"

print({"drive": str(DRIVE_ROOT), "output": str(OUTPUT)})

In [ ]:
import sys
from subprocess import check_call

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab; using paths already set above.")

if not REPO_DIR.exists():
    check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
os.chdir(REPO_DIR)
check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])

In [ ]:
from leaseguard.evaluation.cli import main as evaluation_main

runs = sorted(REPORT_DIR.glob("*-schema-full-4bit.json"))
print("required runs found:", [path.name for path in runs])
if not runs:
    raise SystemExit(
        "No schema-full-4bit runs in Drive. Finish the three candidate notebooks first."
    )

argv = ["compare-baselines", *[str(path) for path in runs], "--output", str(OUTPUT)]
status = evaluation_main(argv)
raise SystemExit(status)